In [10]:
import threading
import time

def print_name(name, *args):
    time.sleep(1)
    print(name, *args)

name1 = "1Lx tutorial ... "
name2 = "2Lx tutorial ... "
name3 = "3Lx tutorial ... "
    
t1 = threading.Thread(target=print_name, args=(name1,1,2))
t2 = threading.Thread(target=print_name, args=(name2,1,2,3))
t3 = threading.Thread(target=print_name, args=(name3,1,2,3,4))

t1.start() # start() = 创建新线程 + 自动调用 run()
t2.start()
t3.start()
"""
t.start()   # ✅ 多线程（并发）
t.run()     # ❌ 单线程（同步）
"""

t1.join()  # 👉 让主线程阻塞，直到子线程执行完
t2.join()
t3.join()

print("All threads started")

1Lx tutorial ... 2Lx tutorial ...  1 2 3
3Lx tutorial ...  1 2 3 4
 1 2
All threads started


对比不使用start 而改成 run 的效果：

In [11]:
import threading
import time

def print_name(name, *args):
    time.sleep(1)
    print(name, *args)

name1 = "1Lx tutorial ... "
name2 = "2Lx tutorial ... "
name3 = "3Lx tutorial ... "
    
t1 = threading.Thread(target=print_name, args=(name1,1,2))
t2 = threading.Thread(target=print_name, args=(name2,1,2,3))
t3 = threading.Thread(target=print_name, args=(name3,1,2,3,4))

t1.run() 
t2.run()
t3.run()

t1.join()  # 👉 让主线程阻塞，直到子线程执行完, 但此处或报错，因为 run 不会启动新的thread
t2.join()
t3.join()

print("All threads started")

1Lx tutorial ...  1 2
2Lx tutorial ...  1 2 3
3Lx tutorial ...  1 2 3 4


RuntimeError: cannot join thread before it is started

In [8]:
import threading
import time

def print_name(name, *args):
    time.sleep(1)
    print(name, *args)

name1 = "1Lx tutorial ... "
name2 = "2Lx tutorial ... "
name3 = "3Lx tutorial ... "
    
t1 = threading.Thread(target=print_name, args=(name1,1,2))
t2 = threading.Thread(target=print_name, args=(name2,1,2,3))
t3 = threading.Thread(target=print_name, args=(name3,1,2,3,4))

t1.run() 
t2.run()
t3.run()

print("All threads started")

1Lx tutorial ...  1 2
2Lx tutorial ...  1 2 3
3Lx tutorial ...  1 2 3 4
All threads started


不使用 join

In [9]:
import threading
import time

def print_name(name, *args):
    time.sleep(1)
    print(name, *args)

name1 = "1Lx tutorial ... "
name2 = "2Lx tutorial ... "
name3 = "3Lx tutorial ... "
    
t1 = threading.Thread(target=print_name, args=(name1,1,2))
t2 = threading.Thread(target=print_name, args=(name2,1,2,3))
t3 = threading.Thread(target=print_name, args=(name3,1,2,3,4))

t1.start() # start() = 创建新线程 + 自动调用 run()
t2.start()
t3.start()
"""
t.start()   # ✅ 多线程（并发）
t.run()     # ❌ 单线程（同步）
"""

# t1.join()  # 👉 让主线程阻塞，直到子线程执行完
# t2.join()
# t3.join()

print("All threads started")

All threads started


1Lx tutorial ... 2Lx tutorial ...  1 2 3
3Lx tutorial ...  1 2 3 4
 1 2


上述输出说明 碰到**线程调度 + 输出交错（interleaving）**的核心问题

本质原因是两点

- 1. 主线程先执行完
- 2. 子线程“同时醒来”，抢占 stdout

---

真实发生的执行顺序（模拟）

我们模拟一个可能的执行顺序：

---

```
🧵 Step 1：t1 开始打印

t1: print("1Lx tutorial ...", 1, 2)
输出了一半：
"1Lx tutorial ..."

🧵 Step 2：CPU切走 → t2 抢到执行权

t2: print("2Lx tutorial ...", 1, 2, 3)
输出：
"2Lx tutorial ... 1 2 3"

👉 于是你看到：

1Lx tutorial ... 2Lx tutorial ...  1 2 3

🧵 Step 3：t3 输出

3Lx tutorial ...  1 2 3 4

🧵 Step 4：t1 继续补完剩下的

 1 2
```

---
 
🎯 关键总结（非常重要）

你的问题本质是：

多个线程同时执行 print，输出被打断并交错，导致一行内容被拆开


## threading life cycle

In [11]:
import threading

def func(x):
    current_thread = threading.current_thread()
    print('Current thread:', current_thread)
    for n in range(x):
        print('{} Running...'.format(current_thread.name), n)

    print("Internal Thread Finished...")

# Create two threads
t1 = threading.Thread(target=func, args=(2,))
t2 = threading.Thread(target=func, args=(3,))

# Start the threads 
t1.start()
t2.start()

# wait for the threads to finish
t1.join()
t2.join()

# simulate some work in the main thread
for n in range(3):
    print('Main Thread Running...', n)

print("Main Thread Finished...")

Current thread: <Thread(Thread-119 (func), started 6292451328)>
Thread-119 (func) Running... 0
Thread-119 (func) Running... 1
Internal Thread Finished...
Current thread: <Thread(Thread-120 (func), started 6309277696)>
Thread-120 (func) Running... 0
Thread-120 (func) Running... 1
Thread-120 (func) Running... 2
Internal Thread Finished...
Main Thread Running... 0
Main Thread Running... 1
Main Thread Running... 2
Main Thread Finished...


## Synchronization primitive

线程同步

In [ ]:
import threading
import time

# Create a semaphore
semaphores = threading.Semaphore(2)  # 允许同时访问的线程数量为2

def worker():

    with semaphores: # 使用 with 语句自动获取和释放信号量
        print("{} has started working...".format(threading.current_thread().name))
        time.sleep(2) # 模拟工作
        print("{} has finished working...".format(threading.current_thread().name))
    
    print("-----------")
    
# Create a list to keep track of thread objects
threads = []
# Create 6 threads
for i in range(6):
    t = threading.Thread(target=worker, name=f"Worker-{i+1}")
    threads.append(t)
    print(f"Starting {t.name}...\n")
    
    t.start()

# Wait for all threads to finish
for t in threads:
    t.join()
print("All workers have finished.")


Starting Worker-1...

Worker-1 has started working...
Starting Worker-2...

Starting Worker-3...

Starting Worker-4...

Starting Worker-5...

Starting Worker-6...

Worker-1 has finished working...
-----------
Worker-2 has started working...
Worker-2 has finished working...
-----------
Worker-3 has started working...
Worker-3 has finished working...
-----------
Worker-4 has started working...
Worker-4 has finished working...
-----------
Worker-5 has started working...
Worker-5 has finished working...
-----------
Worker-6 has started working...
Worker-6 has finished working...
-----------
All workers have finished.
